In [91]:
import pandas as pd
import requests

### Get md file from SpeedyApply github repo for US based Data Science internships

In [92]:
url = "https://raw.githubusercontent.com/speedyapply/2026-AI-College-Jobs/main/README.md"
response = requests.get(url)

with open('raw_internships.md', 'w', encoding='utf-8') as f:
    f.write(response.text)

### Extract table from raw_internships.md

In [93]:
# Helper function
import re
from io import StringIO

def extract_md_table(md_text, start_marker, end_marker):
    pattern = rf"{re.escape(start_marker)}\n(.*?)\n{re.escape(end_marker)}"
    match = re.search(pattern, md_text, flags=re.DOTALL)

    if not match:
        return None

    table_md = match.group(1).strip()

    df = pd.read_csv(
        StringIO(table_md),
        sep="|",
        engine="python"
    )

    # clean up
    df = df.dropna(axis=1, how="all")
    df.columns = df.columns.str.strip()
    df = df.iloc[1:]  # remove markdown separator row
    df = df.apply(lambda x: x.str.strip())

    # handle company link and name
    if "Company" in df.columns:
        df[["Company Link", "Company"]] = df["Company"].str.extract(
            r'href="([^"]+)".*?<strong>(.*?)</strong>'
        )

    # cleanup position link
    if "Posting" in df.columns:
        df["Posting"] = df["Posting"].str.extract(r'href="([^"]+)"')

    return df


In [94]:
# Load the markdown file
with open("raw_internships.md", "r") as f:
    md_text = f.read()

In [95]:
# Extract tables into useable dataframes
faang_df = extract_md_table(
    md_text,
    "<!-- TABLE_FAANG_START -->",
    "<!-- TABLE_FAANG_END -->"
)

quant_df = extract_md_table(
    md_text,
    "<!-- TABLE_QUANT_START -->",
    "<!-- TABLE_QUANT_END -->"
)

other_df = extract_md_table(
    md_text,
    "<!-- TABLE_START -->",
    "<!-- TABLE_END -->"
)


In [96]:
faang_df.head()

,Company,Position,Location,Salary,Posting,Age,Company Link
1,Meta,Research Scientist Intern - Optical Fiber Sens...,"Redmond, WA",$50/hr,https://www.metacareers.com/jobs/1534507591220568,0d,https://about.meta.com
2,Meta,Research Scientist Intern - Robotics Control P...,"Redmond, WA",$50/hr,https://www.metacareers.com/jobs/1556614218924991,0d,https://about.meta.com
3,Roblox,2026 Data Scientist - PhD Intern - Short Term,"San Mateo, CA, United States",$64/hr,https://careers.roblox.com/jobs/7540083?gh_jid...,0d,https://www.roblox.com
4,Meta,Research Scientist Intern - Meta Recommendatio...,"Bellevue, WA",$50/hr,https://www.metacareers.com/jobs/887534076970830,4d,https://about.meta.com
5,Adobe,2026 Intern - Machine Learning Engineer,"San Francisco, CA",$55/hr,https://adobe.wd5.myworkdayjobs.com/en-US/exte...,6d,https://www.adobe.com


In [97]:
quant_df.head()

,Company,Position,Location,Salary,Posting,Age,Company Link
1,Citadel,Machine Learning Researcher - PhD Intern - US,New York,$125/hr,https://www.citadel.com/careers/details/machin...,0d,https://www.citadel.com/careers
2,Citadel,Quantitative Research Analyst Intern - BS/MS - US,New York,$125/hr,https://www.citadel.com/careers/details/quanti...,0d,https://www.citadel.com/careers
3,Citadel,Quantitative Researcher - PhD Intern - US,New York,$125/hr,https://www.citadel.com/careers/details/quanti...,0d,https://www.citadel.com/careers
4,Citadel,Quantitative Research Engineer - PhD Intern - US,New York,$125/hr,https://www.citadel.com/careers/details/quanti...,0d,https://www.citadel.com/careers
5,Citadel Securities,Quantitative Researcher - PhD Intern - US,New York,$125/hr,https://www.citadelsecurities.com/careers/deta...,0d,https://www.citadelsecurities.com/careers


In [98]:
other_df.head()

,Company,Position,Location,Posting,Age,Company Link
1,Brain Co.,AI Product 2026 Internship,San Francisco Bay Area,https://jobs.ashbyhq.com/brainco/dc0e574f-8a1b...,0d,https://www.braincompany.ai/
2,Brain Co.,AI/ML 2026 Internship,San Francisco Bay Area,https://jobs.ashbyhq.com/brainco/0abea2f1-324e...,0d,https://www.braincompany.ai/
3,CLEAR,Member Care AI Enablement Intern - Summer 2026,"New York, New York, United States",https://job-boards.greenhouse.io/clear/jobs/75...,0d,https://www.clearme.com
4,Instacart,Machine Learning Engineer - PhD Intern,Remote,https://instacart.careers/job/?gh_jid=5917202,0d,http://instacart.com
5,JLL,AI Development Intern 2026 - Chicago,USA CORP Chicago IL Chicago,https://jll.wd1.myworkdayjobs.com/en-US/jllcar...,0d,https://www.us.jll.com


### Clean the Dataframes

In [99]:
print("Faang:", faang_df.shape)
print("Quant:", quant_df.shape)
print("Other:", other_df.shape)

Faang: (89, 7)
Quant: (8, 7)
Other: (266, 6)


In [100]:
other_df.isna().sum()

Company         0
Position        0
Location        0
Posting         0
Age             0
Company Link    0
dtype: int64

In [101]:
other_df.columns

Index(['Company', 'Position', 'Location', 'Posting', 'Age', 'Company Link'], dtype='object')

In [ ]:
other_df.Company.unique().size
# 169 unique companies from 266 job postings

169

In [108]:
faang_df.Company.unique().size
# 9 faang companies with 89 job postings

9

In [ ]:
quant_df.Company.unique().size
# 3 quant companies with 8 total postings

3